# Data Modelling Automation Toolkit
Version 0.12 | September 2025  
Nicholas Ballingall

#### Project Desc.
This proof-of-concept toolkit leverages Google's Gemini large language model (among other methods) to automate several key productivity bottlenecks faced by the Data Modelling team at Lloyds Banking Group (LBG):
1. Foundational dataset profiling
2. Writing attribute descriptions
3. Applying a standard attribute naming convention
4. Schema normalisation (part-automated)

## Toolkit Logic Pipeline

### 0. Notebook Setup

In [63]:
# --- 0.1 Import Libraries ---

# Data Manipulation
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import json
import re
from typing import Literal

# Data Validation
import enum
from typing import Optional, List
from pydantic import BaseModel, Field, field_validator, ValidationInfo, ValidationError, model_validator, computed_field

# File Handling
import os
import glob
from pathlib import Path # platform-agnostic path management
import chardet
import zipfile
from datetime import datetime

# XML Formatting
from xml.etree.ElementTree import Element, SubElement, tostring
from xml.etree.ElementTree import parse as file_parse
from xml.dom import minidom

# Google genAI SDK
from google import genai
from google.genai import types
from google.genai.errors import ServerError # for 503 error exponential backoff

# Misc.
import time
import functools

In [64]:
# --- 0.2 Initialise GenAI Client ---

# [Declaring API key directly for testing]
client = genai.Client(api_key = 'YOUR_API_KEY_HERE')

In [65]:
# --- 0.3 Misc. Code Formatting ---

# Set print indent to 3 spaces
ind = '   '

### 1. Data Preperation & Analysis

In [67]:
# --- 1.1 File Reading Utility ---

def dataset_reader(filename, decode_conf_threshold = 0.9, in_data_dir=True, show_progress=False):
    """
    Detects encoding for CSVs or reads Excel files.

    Args:
        - filename (str): name of the '.csv', '.xlsx', or '.xls' file to be read
        - decode_conf_threshold (float): Confidence threshold for chardet encoding detection (0.9 default)
        - in_data_dir (bool): If true, assumes the file is in the standard run/project data directory.
                              Otherwise, 'filename' must include the full file path.
        - show_progres (bool): If true, print process progress

    Returns:
        - DataFrame: df of excel/csv data
    """
    # -- Extract/validate file extension --
    ext = os.path.splitext(filename)[-1].lower()
    if ext in ['.csv', '.xlsx', '.xls']:
        
        # Validate filepath
        if in_data_dir:
            # If file is in the default data directory, join the paths
            file = os.path.join(data_dir, filename)
        else:
            # If not, the filename must be an absolute path
            if not os.path.isabs(filename):
                raise ValueError(
                    "Expected an absolute filepath for 'filename' when 'in_data_dir' is False, "
                    f"but got a relative path:\n   > '{filename}'"
                )
            file = filename
    else:
        raise ValueError(f"Unsupported file type: {ext}\n   > Can only use '.csv', '.xlsx', or '.xls' files")

    # -- Process CSV files --
    if ext == '.csv':
        # Read data sample to detect encoding
        with open(file, 'rb') as f:
            sample = f.read(100_000) # 100kb sample size
            det = chardet.detect(sample)
            encoding, conf = det.get('encoding'), det.get('confidence') or 0.0

        if show_progress:
            print(f"Reading {os.path.basename(file)}...")
            print(f"{ind}> Detected CSV encoding: {encoding} ({conf:.0%} confidence)")
        
        if not encoding or conf < decode_conf_threshold:
            if show_progress:
                print(f"{ind}> Low confidence in CSV encoding detection – defaulting to UTF-8")
            encoding = 'utf-8'

        # Return DataFrame of csv data ('sep=None' autodetects seperator)
        return pd.read_csv(file, encoding=encoding, engine='python', sep=None, comment='#')

    # -- Process Excel files --
    else:
        return pd.read_excel(filename, engine='openpyxl')

In [68]:
# --- 1.2 Dataset Profiling ---

def _profile_column(df: pd.DataFrame, column_name: str) -> dict:
    """
    Analyzes a single DataFrame column to create a detailed data profile.
    Creates a robust structure for categorical stats to prevent XML errors.
    """
    series = df[column_name]
    
    # -- Universal metrics (calculated once for efficiency) --
    total_count = len(series)
    null_count = series.isnull().sum()
    non_null_series = series.dropna()
    unique_values = non_null_series.unique()
    unique_count = len(unique_values)

    # --- Convert NumPy types ---
    def convert_numpy_types(val):
        if isinstance(val, np.integer):
            return int(val)
        if isinstance(val, np.floating):
            return float(val)
        if isinstance(val, np.bool_):
            return bool(val)
        return val

    profile = {
        "column_name": column_name,
        "data_type": str(series.dtype),
        "completeness_stats": {
            "total_records": total_count,
            "null_count": int(null_count),
            "null_percentage": round((null_count / total_count) * 100, 2) if total_count > 0 else 0
        },
        "cardinality_stats": {
            "unique_count": unique_count,
            "unique_percentage": round((unique_count / total_count) * 100, 2) if total_count > 0 else 0
        },
        "sample_values": [convert_numpy_types(val) for val in unique_values[:5]]
    }

    # -- Type-specific descriptive statistics --
    if pd.api.types.is_numeric_dtype(series):
        stats = non_null_series.describe()
        profile["numeric_stats"] = {
            "min": float(stats.get("min", 0)),
            "max": float(stats.get("max", 0)),
            "mean": float(stats.get("mean", 0)),
            "median": float(stats.get("50%", 0)),
            "std_dev": float(stats.get("std", 0)),
            "q1_25th_percentile": float(stats.get("25%", 0)),
            "q3_75th_percentile": float(stats.get("75%", 0)),
        }
    elif pd.api.types.is_datetime64_any_dtype(series):
        profile["datetime_stats"] = {
            "min_date": str(non_null_series.min()),
            "max_date": str(non_null_series.max())
        }
    elif pd.api.types.is_string_dtype(series) or isinstance(series.dtype, CategoricalDtype):
        value_counts = non_null_series.value_counts().nlargest(5)
        profile["categorical_stats"] = {
            "most_frequent_values": [
                {"value": str(val), "count": int(count)}
                for val, count in value_counts.items()
            ]
        }
        # String length analysis remains the same
        str_lengths = non_null_series.astype(str).str.len()
        length_stats = str_lengths.describe()
        profile["string_stats"] = {
            "min_length": int(length_stats.get("min", 0)),
            "max_length": int(length_stats.get("max", 0)),
            "avg_length": round(length_stats.get("mean", 0), 2)
        }

    return profile

def dataset_profiler(
    data_df: pd.DataFrame, 
    desc_df: pd.DataFrame = None,
    desc_name_col: str = 'attribute_name',
    desc_text_col: str = 'business_description'
) -> list:
    """
    Produces LLM-readable attribute profiles for a given dataset.
    If a descriptions DataFrame ('desc_df') is provided, it will be merged.

    Args:
        - data_df (DataFrame): the input dataset
        - desc_df (DataFrame, optional): attribute descriptions – defaults to None
        - desc_name_col (str, optional): the 'desc_df' column containing the attribute name
        - desc_text_col (str, optional): the 'desc_df' column containing the description text
    
    Returns:
        - list: a list of consolidated and detailed attribute profiles (dicts)
    """
    consolidated_profiles = []
    desc_map = {}

    print(f"{ind}> Profiling dataset...")
    
    # -- Create description map from the descriptions DataFrame (optional) --
    if desc_df is not None:
        try:
            desc_map = dict(zip(desc_df[desc_name_col], desc_df[desc_text_col]))
        except KeyError as e:
            print(f"{ind}>ERROR: Could not process descriptions DataFrame due to missing column/n   > Error: {e}")
        except Exception as e:
            print(f"{ind}>ERROR: An unexpected error occurred while processing descriptions/n   > Error: {e}")
    
    # -- Generate a detailed profile for each column --
    for col in data_df.columns:
        profile = _profile_column(data_df, col)
        
        # Add description to the profile if available
        profile['description'] = desc_map.get(col, '-') # '-' when no description provided
        consolidated_profiles.append(profile)

    return consolidated_profiles

### 2. Prompt Engineering Utilities

In [70]:
# --- 2.1 Prompt Text Formatting ---

class TextFormatter:
    """
    Converts the formatting of a string object from Markdown or JSON to XML,
    JSON, Markdown, or plaintext. All formats have some level of structure, with
    plaintext and markdown closely matching human writing. JSON and XML strings
    have more sophisticated and apparent structure.
    """
    def __init__(self, input_data):
        """
        Initialises the formatter with the source string objects (formatted in
        markdown or as JSON-strings)

        Args:
            - input_data (str): The string object to be formatted
        """
        self.original_input = input_data
        self.json_data = None
        self.md_text = ""

        if isinstance(input_data, (list, dict)):
            self.json_data = input_data
            self.md_text = self._json_to_markdown(self.json_data)
        elif isinstance(input_data, str):
            try:
                self.json_data = json.loads(input_data)
                self.md_text = self._json_to_markdown(self.json_data)
            except json.JSONDecodeError:
                self.md_text = input_data # Input is Markdown
        else:
            raise TypeError("Input must be a string, list, or dictionary.")

    # -- Convert to XML --
    def _sanitize_tag(self, text: str) -> str:
        """
        Strips leading/trailing whitespace, forces lowercase, and swaps spaces
        for underscores in a string object to make a suitable XML tag.
        """
        return text.strip().lower().replace(' ', '_')

    def _singularize(self, plural: str) -> str:
        """
        Converts a plural noun into its singular form for list-item XML tags.
        If the word ends with 's', it is removed. If not, the generic 'item'
        tag is used.
        """
        if plural.endswith('s'):
            return plural[:-1]
        return 'item'

    def _build_xml_recursive(self, parent_element, data):
        """
        Recursively runs through a dictionary or list to build an XML structure.
        For dictionaries, it creates new sub-elements. For lists, it iterates through
        items, creating a tag for each one.
        
        Will call itself to handle nested structures.

        Args:
            - parent_element (Element): The parent XML element to which new child
                                        elements will be attached.
            - data (dict/list): The data (often nested) to convert into XML.
        """
        # Do not make child tags for these items
        keys_to_skip = [
            'column_name',
            'name',
            'attribute',
            'attribute_name'
        ]
        for key, value in data.items():
            if key in keys_to_skip: # skip defined keys
                continue

            # Create XML tags
            tag_name = self._sanitize_tag(key)
            # Dictionary object
            if isinstance(value, dict):
                # Self-call for child tags
                child_element = SubElement(parent_element, tag_name)
                self._build_xml_recursive(child_element, value)

            # List object
            elif isinstance(value, list):
                item_tag_name = self._singularize(parent_element.tag)
                # Parent tags are pluar, child tags singular
                for item in value:
                    # Use a dynamic tag if column_name exists, otherwise use the singularised parent
                    item_tag = item_tag_name 
                    if isinstance(item, dict) and 'column_name' in item:
                        item_tag = self._sanitize_tag(item['column_name'])
                    
                    # Attach the item directly to the parent element
                    item_element = SubElement(parent_element, item_tag)

                    if isinstance(item, dict):
                        self._build_xml_recursive(item_element, item)
                    else:
                        item_element.text = str(item)
            else:
                child_element = SubElement(parent_element, tag_name)
                child_element.text = str(value)

    def to_xml(self) -> str:
        """
        Converts the stored data into a formatted XML string.

        If the initial input was JSON, it builds the XML from the structured
        data. If the input was Markdown, it performs a basic conversion by
        treating headings as parent tags and list items as child tags.

        Returns:
            str: A pretty-printed XML string.
        """
        # -- JSON String INput --
        if self.json_data:
            record = self.json_data
            root_tag_name = self._sanitize_tag(record.get('name', 'record'))
            root = Element(root_tag_name)
            self._build_xml_recursive(root, record)
            
            rough_string_bytes = tostring(root, 'utf-8')
            reparsed = minidom.parseString(rough_string_bytes)
            xml_string = reparsed.toprettyxml(indent="  ") # pretty-print
            
            return '\n'.join(xml_string.split('\n')[1:])
        
        # -- non-JSON Input --
        else:
            lines = self.md_text.strip().split('\n')
            output_lines = []
            current_parent_tag = None
            for line in lines:
                stripped_line = line.strip()
                if not stripped_line: continue

                # Headings
                heading_match = re.match(r'^(#+)\s+(.*)', stripped_line) # MD-specific syntax
                if heading_match:
                    if current_parent_tag:
                        output_lines.append(f"</{current_parent_tag}>")
                    current_parent_tag = self._sanitize_tag(heading_match.group(2))
                    output_lines.append(f"<{current_parent_tag}>")
                    continue

                # Lists (bulleted)
                list_match = re.match(r'^[*-]\s+(.*)', stripped_line)
                if list_match and current_parent_tag:
                    item_text = list_match.group(1).strip()
                    child_tag = self._singularize(current_parent_tag)
                    output_lines.append(f"  <{child_tag}>{item_text}</{child_tag}>")
                elif current_parent_tag:
                    output_lines.append(f"  {stripped_line}")

            if current_parent_tag:
                output_lines.append(f"</{current_parent_tag}>")
            return "\n".join(output_lines)

    # -- Convert to Markdown --
    def _format_entry(self, key, value, level):
        """
        Recursively formats a key-value pair into a Markdown list entry.

        This helper function handles nested dictionaries and lists to create
        an indented, hierarchical list structure in Markdown.

        Args:
            - key (str): The key to be formatted as the list item's title.
            - value (any): The value to be formatted, which can be a simple type,
              a dictionary, or a list.
            - level (int): The current nesting level for indentation.

        Returns:
            - str: A formatted Markdown string for the entry.
        """
        # Calculate indent level
        indent = "  " * (level - 1)
        line_start = f"{indent}- **{key.replace('_', ' ').title()}**: "

        # -- From dict structure --
        if isinstance(value, dict):
            entry_str = f"{line_start}\n"
            for sub_key, sub_value in value.items():
                entry_str += self._format_entry(sub_key, sub_value, level + 1)
            return entry_str

        # -- From list structure --
        elif isinstance(value, list):
            list_str = ", ".join(map(str, value))
            return f"{line_start}{list_str}\n"
        
        else:
            return f"{line_start}{value}\n"

    def _json_to_markdown(self, data):
        """
        Converts a dictionary or list into a markdown-formatted string.

        It creates a main heading from the data's 'name' or 'column_name'/'attribute_name' and
        then formats all other key-value pairs as a nested list.

        Args:
            - data (dict or list): The JSON object to convert

        Returns:
            - str: A string formatted in markdown
        """
        md_parts = []

        # -- Ensure list formatting -- 
        if not isinstance(data, list): data = [data]

        # Loop through list
        for item in data:
            # -- Assign title --
            title = item.get('column_name', item.get('attribute_name', item.get('name', 'Details')))
            md_parts.append(f"## {title}\n")

            # -- Assign body --
            for key, value in item.items():
                # Filter attribute/column name (already title)
                if key in ['column_name', 'attribute_name', 'name']: continue
                md_parts.append(self._format_entry(key, value, level=1))
        
        return "".join(md_parts)

    def to_markdown(self) -> str:
        """
        Returns the markdown representation of the input data (self.md_text
        attribute from __init__).
        """
        return self.md_text

    # -- Convert to Plaintext --
    def to_plaintext(self) -> str:
        """
        Strips all markdown syntax from an input string, returning the same content
        as a plaintext string. Markdown headings are CAPITALISED in plaintext.

        Returns:
            - str: The plaintext string
        """
        # -- Split input line-by-line --
        md_string = self.to_markdown() # standardise to markdown
        lines = md_string.strip().split('\n')
        output_lines = []

        # -- Remove & replace markdown syntax --
        for line in lines:
            stripped_line = line.strip()
            if not stripped_line:
                continue

            # Titles
            hash_match = re.match(r'^#+\s+(.*)', stripped_line)
            if hash_match:
                output_lines.append(hash_match.group(1).strip().upper()) # capitalise
                continue

            # Bold text
            bold_match = re.match(r'^\*\*(.*)\*\*$', stripped_line)
            if bold_match:
                output_lines.append(bold_match.group(1).strip().upper()) # capitalise
                continue

            # List items
            list_match = re.match(r'^(\s*)[*-]\s+(.*)', line)
            if list_match:
                indent = list_match.group(1) # indent
                content = list_match.group(2).strip()
                output_lines.append(f"{indent}– {content}") # add "-" as stand-in bullet
                continue
            output_lines.append(stripped_line)
        output_text = "\n".join(output_lines)
        return re.sub('[*#]', '', output_text)

    # -- Convert to JSON --
    def to_json(self) -> str:
        """
        Converts the stored data into a formatted JSON string.

        If the original input was JSON, it returns that da ta. If the input was
        markdown, it parses the markdown structure (headings and lists) back into
        a dictionary.

        Returns:
            - str: A pretty-printed JSON string
        """
        # JSON format check
        if self.json_data is not None:
            return json.dumps(self.json_data, indent=2)

        md_string = self.to_markdown() # markdown conversion
        lines = md_string.strip().split('\n')
        json_dict = {} # create dict object
        current_key = None
        current_value = []

        # -- Function to store parsed values in dict object --
        def store_previous_value():
            if current_key and current_value:
                if len(current_value) == 1:
                    json_dict[current_key] = current_value[0]
                else:
                    json_dict[current_key] = current_value

        # -- Structure input as JSON --
        for line in lines:
            stripped_line = line.strip()
            if not stripped_line:
                continue
            # Markdown syntax check
            hash_match = re.match(r'^#+\s+(.*)', stripped_line)
            bold_match = re.match(r'^\*\*(.*)\*\*$', stripped_line)
            is_new_section, heading_text = False, ""

            # -- MD syntax -> JSON structure --
            if hash_match: # titles
                is_new_section, heading_text = True, hash_match.group(1)
            elif bold_match: # bold
                is_new_section, heading_text = True, bold_match.group(1)
            
            if is_new_section:
                store_previous_value()
                current_key = self._sanitize_tag(heading_text)
                current_value = []
                continue
            
            list_match = re.match(r'^[*-]\s+(.*)', stripped_line)
            if list_match: # lists
                if not isinstance(current_value, list):
                    current_value = [current_value]
                current_value.append(list_match.group(1).strip())
            elif current_key:
                current_value.append(stripped_line)
        store_previous_value()
        return json.dumps(json_dict, indent=2)

In [71]:
# --- 2.2 Prompt Builder ---

def prompt_builder(output_format: str, **components) -> str:
    """
    Builds a single formatted prompt string from various named components.

    Each component is formatted (TextFormatter) individually before being
    combined in sequential order. Plaintext strings have titles assigned from
    keywords, whilst structured strings (e.g. markdown) are left to use their
    internal titles.

    Args:
        - output_format (str): the target format ('markdown', 'xml', 'plaintext', 'json')
        - **components: keyword arguments representing individual parts of the prompt

    Returns:
        - str: the final combined & formatted prompt string
    
    Raises:
        - ValueError: if an unsupported output_format is provided
    """
    formatted_parts = []

    # Map 'output_formats' to functions within TextFormatter
    format_map = {
        'json': 'to_json', 'xml': 'to_xml',
        'markdown': 'to_markdown', 'plaintext': 'to_plaintext'
    }
    method_name = format_map.get(output_format.lower())

    # Error for unsupported formats
    if not method_name:
        raise ValueError(f"Unsupported format. Use one of: {', '.join(format_map.keys())}")

    # --- Format each prompt component seperately ---
    for name, data in components.items():
        input_data = None

        # Dictionary input
        if isinstance(data, dict):
            input_data = {'name': name, **data}
        
        # List input
        elif isinstance(data, list):
            input_data = {'name': name, 'items': data}
        
        # String input
        elif isinstance(data, str):
            stripped_data = data.strip()
            
            # --- Handle component title ---
            if stripped_data.startswith('#') or stripped_data.startswith('*'): # MD title syntax
                # Use internal title for structured components
                input_data = data
            else:
                # Assign title from keyword for unstructured components
                title = name.replace('_', ' ').title()
                input_data = f"## {title}\n{stripped_data}"
        else:
            input_data = str(data)

        # --- Format each component w/ TextFormatter ---
        formatter = TextFormatter(input_data)
        conversion_method = getattr(formatter, method_name)
        formatted_text = conversion_method().strip()
        formatted_parts.append(formatted_text)

    # --- Combine components into single prompt ---
    combined_body = "\n\n".join(formatted_parts)

    # Add a root element for XML output
    if output_format.lower() == 'xml':
        return f"<prompt>\n{combined_body}\n</prompt>"
    else:
        return combined_body

### 3. LLM Calling Utilities

In [73]:
# --- 3.1 LLM Calling Utility Functions ---

def exponential_backoff_retry(max_retries: int, initial_delay: int = 2):
    """
    A decorator to retry a function call with exponential backoff on 503 ServerError.

    Args:
        - max_retries (int): maximum number of retry attempts
        - initial_delay (int): waiting time for the first retry in seconds
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            delay = initial_delay
            last_exception = None
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                
                except ServerError as e:
                    last_exception = e
                    # Convert the error to a string and check its content directly.
                    if str(e).startswith('503'):
                        
                        print(f"{ind}> Attempt {attempt + 1}/{max_retries} failed with 503 Server Error.")
                        
                        if attempt == max_retries - 1:
                            print("{ind}> Max retries reached. Raising the final error.")
                            raise last_exception
                        
                        print(f"{ind}{ind}>> Retrying in {delay} seconds...")
                        time.sleep(delay)
                        delay *= 2
                    else:
                        # If it's a different ServerError, raise it immediately
                        raise
            
        return wrapper
    return decorator

In [74]:
# --- 3.2 LLM API Calling ---

# Apply exponential backoff retry decorator
@exponential_backoff_retry(max_retries=4)
def generate_output(
    genai_model, prompt, config,
    return_df=False, return_metadata=False, return_response_text=False, print_output=False
):
    """
    Sends formatted prompt + configuration settings to LLM, returns a validated response.

    Args:
        - genai_model (str): name of the [Google] model to be used
        - prompt (str): prompt to be sent to the model
        - config (dict): dictionary of LLM configuration settings, may include 'validation_context'
        
        - return_df (bool): If True, returns the parsed data as a DataFrame.
        - return_metadata (bool): If True, returns a dictionary of call metadata.
        - return_response_text (bool): If True, returns the raw text response from the LLM.
        - print_output (bool): If True, prints the metadata and raw response text to the console.

    Returns:
        - any or tuple: By default, returns the parsed data (dict or DataFrame). If any 'return_*'
                        flags are set, returns a tuple containing the requested items in the order:
                        (parsed_data, metadata, response_text).
    """
    # Initialise 'response' as None – prevents 'UnboundLocalError'
    response = None
    
    # --- Separate the dictionary keys from the API config ---
    custom_keys = ['name', 'validation_context']
    api_config = config.copy()
    for key in custom_keys:
        api_config.pop(key, None) # Use .pop with a default to avoid errors
    
    validation_context = config.get('validation_context')
    schema_class = config.get('response_schema')

    try:
        # --- Call LLM ---
        response = client.models.generate_content(
            model=genai_model,
            contents=prompt,
            config=api_config
        )
        response_text = response.text
        
    except Exception as e:
        print(f"ERROR: LLM call failed. Error: {e}")
        # On failure, return empty/None values based on what was requested
        return_values = [pd.DataFrame() if return_df else {}]
        if return_metadata:
            return_values.append(None)
        if return_response_text:
            return_values.append(None)
        return tuple(return_values) if len(return_values) > 1 else return_values[0]

    # --- Handle Metadata and Printing ---
    metadata_obj = None
    if return_metadata or print_output:
        prompt_tokens = client.models.count_tokens(model=genai_model, contents=prompt)
        metadata_obj = {
            'prompt_tokens': prompt_tokens.total_tokens,
            'response_usage_metadata': response.usage_metadata
        }

    if print_output:
        print("\n--- LLM Call Metadata ---")
        if metadata_obj:
            print(f"{ind}> Prompt Tokens: {metadata_obj['prompt_tokens']}")
            print(f"{ind}> Response Usage Metadata:\n{metadata_obj['response_usage_metadata']}")
        print(f"\n--- LLM Response Text ---\n{response_text}\n-------------------------")

    # --- Parse LLM response ---
    parsed_output = None
    if schema_class:
        try:
            # Use regex to find the JSON object, stripping any potential wrapping
            json_match = re.search(r"\{.*\}", response_text, re.DOTALL)
            if not json_match:
                raise ValueError("No JSON object found in the response.")
            json_str = json_match.group(0)
            
            schema_instance = schema_class.model_validate_json(json_str, context=validation_context)
        
            if return_df:
                response_data = schema_instance.model_dump()
                # Dataframe creation logic
                if isinstance(schema_instance, SuffixSchema):
                    parsed_output = pd.DataFrame(response_data['suffixes'])
                else:
                    parsed_output = pd.DataFrame(response_data)
            else:
                # When not returning a Dataframe, return the validated Pydantic object itself
                parsed_output = schema_instance
                
        except Exception as e:
            print(f"--- FAILED PARSE: Full Response Text ---\n{response_text}")
            print(f"--- PARSE ERROR ---\n{e}")
            parsed_output = pd.DataFrame() if return_df else {}
    else:
        # If no schema, the "parsed" output is just the raw text
        parsed_output = response_text

    # --- Assemble and Return Final Output(s) ---
    return_values = [parsed_output]
    if return_metadata:
        return_values.append(metadata_obj)
    if return_response_text:
        return_values.append(response_text)

    # Return a single item if that's all that was requested, otherwise return a tuple
    if len(return_values) == 1:
        return return_values[0]
    else:
        return tuple(return_values)

In [75]:
# --- 3.3 Final LLM Response Validation ---

# --- Schema primary/foreign key validation (in LLM response) ---
class AttributeKey(enum.Enum):
    PK = "Primary Key"
    FK = "Foreign Key"
    NK = "Not Key"

# Backup validation for acronyms, contractions, and abbreviations in LLM response
standardization_map = {
    'ID': 'Identifier',
    'NUM': 'Number'
}

class AttributeName(BaseModel):
    """
    A Pydantic model to validate & standardize attribute name construction.
    
    All attribute names must be formatted as: [Entity Subject Qualifier Suffix],
    with the latter two being contextually optional.
    """
    entity: str
    subject: str
    qualifier: Optional[str] = None
    suffix: Optional[str] = None

    # --- Standardization & text case validatior ---
    @field_validator('entity', 'subject', 'qualifier')
    def validate_form(cls, v: str):
        """
        Standardizes known aliases and validates that each word in the phrase
        is either in Proper Case or a valid acronym.
        
        Does not operate on suffixes.
        """
        if not v:
            return v
    
        standardized_v = standardization_map.get(v.upper(), v)
        words = standardized_v.split(' ')
    
        for word in words:
            is_title_case = (word == word.title())
            is_acronym = (word.isupper() and len(word) <= 5)
            
            # If the word is valid as-is, continue
            if is_title_case or is_acronym:
                continue
    
            # If word failed, try splitting by capital letters (PascalCase)
            sub_words = re.findall('[A-Z][^A-Z]*', word)
            
            # If not PascalCase, it's a true failure
            if not sub_words or sub_words == [word]:
                 raise ValueError(f"The word '{word}' in '{standardized_v}' must be in Proper Case or be a valid acronym.")
    
            # Validate each of the sub-words
            for sub in sub_words:
                if not (sub == sub.title() or (sub.isupper() and len(sub) <= 5)):
                    raise ValueError(f"The component '{sub}' in the PascalCase word '{word}' is not valid.")
                
        return standardized_v

    @field_validator('suffix')
    def validate_suffix(cls, v: str, info: ValidationInfo):
        """
        Standardizes the suffix formatting and ensures is it from the list
        provided in the validation context.
        """
        if not v:
            return v
        
        # Check if the context with allowed suffixes was provided
        if not info.context or 'allowed_suffixes' not in info.context:
            # Failsafe in case the context isn't passed
            raise ValueError("Validation context with 'allowed_suffixes' was not provided.")

        standardized_v = standardization_map.get(v.upper(), v)
        allowed_suffixes = info.context['allowed_suffixes']
    
        # Check lowercase suffix
        if standardized_v.lower() not in allowed_suffixes:
            raise ValueError(f"'{standardized_v}' (from input '{v}') is not in the list of allowed suffixes.")
    
        return standardized_v

    @model_validator(mode='after')
    def omit_redundant_suffix(self) -> 'AttributeName':
        """
        Checks if the suffix is redundant and removes it if so.
        """
        # Same logic, but fields accessed with 'self'
        if self.suffix:
            if self.suffix == self.subject:
                print(f"{ind}> Redundant suffix – '{self.suffix}' matches subject. Removing.")
                self.suffix = None
            elif self.qualifier and self.suffix == self.qualifier:
                print(f"{ind}> Redundant suffix – '{self.suffix}' matches qualifier. Removing.")
                self.suffix = None
        
        # The method must return the model instance (`self`)
        return self

    @computed_field
    @property
    def build_full_name(self) -> str:
        """
        Constructs the final attribute name from its components.
        """
        parts = [self.entity, self.subject]
        if self.qualifier:
            parts.append(self.qualifier)
        if self.suffix:
            parts.append(self.suffix)
        return " ".join(parts)

# --- Final attribute validation ---
class Attribute(BaseModel):
    """
    Validation for returning attributes:
        - attribute_name: follows naming rules
        - attribute_key: only contains [FK, PK, NK] values.
    """
    attribute_name: AttributeName
    attribute_key: AttributeKey

# --- Entity creation validation ---
class Entity(BaseModel):
    """
    Each entity consists of a name and a list of its attributes. This model
    enforces fundamental database rules:
        - Each entity must have a name
        - Each entity needs >= 1 attribute
        – Each entity needs == 1 primary key
    """
    entity_name: str = Field(
        ..., 
        min_length=1, 
        description = "The name of the entity (table). Cannot be empty."
    )
    # Use a plural name for the list & ensure it's not empty
    attributes: list[Attribute] = Field(
        ..., 
        min_items=1,
        description = "An entity must contain at least one attribute."
    )

    @field_validator('attributes')
    def validate_single_primary_key(cls, attributes):
        """
        Ensures the entity contains exactly one Primary Key (PK).

        This Pydantic validator checks the list of attributes to confirm
        that ONLY one is designated as the Primary Key.
        """
        primary_key_count = sum(
            1 for attr in attributes if attr.attribute_key == AttributeKey.PK
        )
        
        if primary_key_count == 0:
            raise ValueError("The entity must have a Primary Key.")
        if primary_key_count > 1:
            raise ValueError("The entity cannot have more than one Primary Key.")
            
        return attributes

class NormalizedSchema(BaseModel):
    """
    Represents the complete, normalised database schema.

    This is the top-level container model that holds a list of all the
    individual entities that make up the schema.
    """
    entities: list[Entity]

In [76]:
# --- 3.4 Draft LLM Response Validation ---

class DraftAttributeName(BaseModel):
    """
    A less-strict version of AttributeName that omits the suffix validation.
    It still performs standardization and case validation.
    """
    entity: str
    subject: str
    qualifier: Optional[str] = None
    suffix: Optional[str] = None

    @field_validator('entity', 'subject', 'qualifier', 'suffix')
    def _validate_case(cls, v: str):
        """
        Standardizes known aliases and enforces Proper Case on the output,
        while leaving short acronyms untouched.
        """
        if not v:
            return v
            
        # Standardize known aliases
        standardized_v = standardization_map.get(v.upper(), v)
        
        # Check if the value is a short acronym (e.g., NASA, PSCP) – if so, keep uppercase
        if standardized_v.isupper() and len(standardized_v) <= 5: # 5-character acronym max
            return standardized_v
        
        # For all other cases, enforce the .title() format
        return standardized_v.title()
    
    # Clean up final object
    @model_validator(mode='after')
    def omit_redundant_suffix(self) -> 'DraftAttributeName':
        if self.suffix and (self.suffix == self.subject or self.suffix == self.qualifier):
            self.suffix = None
        return self

class DraftAttribute(BaseModel):
    attribute_name: DraftAttributeName
    attribute_key: AttributeKey

class DraftEntity(BaseModel):
    entity_name: str
    attributes: list[DraftAttribute]

class DraftNormalizedSchema(BaseModel):
    entities: list[DraftEntity]

In [77]:
# --- 3.5 Pipeline Step Schemas ---

class SuffixDetail(BaseModel):
    """
    A sub-model for the details of a single suffix.
    """
    suffix: str
    logical_data_type: str
    description: str

class SuffixSchema(BaseModel):
    """
    The main schema, expecting a list of SuffixDetail objects.
    """
    suffixes: list[SuffixDetail]

class AttributeSchema(BaseModel):
    new_attribute_name: list[str]
    old_column_name: list[str]

class DescriptionSchema(BaseModel):
    attribute_name: list[str]
    business_description: list[str]

### 4. Data Modelling Toolkit

In [79]:
# --- 4.1 Modelling Toolkit Utilities ---

# --- Save Final Outputs ---
def save_output_csv(
    output_dir: str,
    model_name: str,
    prompt_format: str,
    config_name: str,
    results_dict: dict
):
    """
    Saves the DataFrame contents from the 'processed_data' key in the
    results dictionary to uniquely named CSV files.
    """
    # Target the df dictionary
    data_to_save = results_dict.get('processed_data')

    if not isinstance(data_to_save, dict):
        print(f"{ind}> No 'processed_data' found to save.")
        return

    # Iterate through the items in df dictionary
    for key, value in data_to_save.items():
        if isinstance(value, pd.DataFrame) and not value.empty:
            # Create a descriptive, unique filename
            file_name = f"{key}_{model_name}_{prompt_format}_{config_name}.csv"
            file_path = os.path.join(output_dir, file_name)
            try:
                value.to_csv(file_path, index=False)
                print(f"{ind}> Saved '{file_name}'")
            except Exception as e:
                print(f"{ind}> FAILED to save '{file_name}'. Error: {e}")

In [80]:
# --- 4.2 Manage Schema (LDM) Components ---

def time_step(step_func):
    """
    Decorator to track the response times of each pipeline step.
    """
    @functools.wraps(step_func)
    def wrapper(self, *args, **kwargs): # The wrapper will receive 'self' when called
        step_name = step_func.__name__
        start_time = time.time()
        
        # Call the original method, passing 'self' along with other args
        result = step_func(self, *args, **kwargs)
        
        end_time = time.time()
        duration = round(end_time - start_time, 2)
        
        # Access 'self.timings' from the instance the method was called on
        self.timings[step_name] = duration
        print(f"{ind}> '{step_name}' completed in {duration}s")
        
        return result
    return wrapper

class SchemaManager:
    """
    Manages the lifecycle of schema components by orchestrating LLM calls.

    Handles the sequential process of generating suffixes, standardised attribute
    names, descriptions, and optionally creating a normalized 3NF schema.
    
    Maintains internal dictionaries to store the results and debug data of each step.
    """
    def __init__(
        self,
        model_name: str, base_config: dict,
        profiler_func, builder_func, caller_func,
        dev_mode: bool = True):
        """
        Initialises the SchemaManager.

        Args:
            - model_name (str): name of the GenAI model to use
            - base_config (dict): A dictionary with default generation settings
            - profiler_func: function used to profile a DataFrame (e.g., dataset_profiler)
            - builder_func: [prompt_builder] function used to produce final prompts
            - caller_func: function used to call the LLM and get output
            - dev_mode (bool): If True, requests and stores metadata and raw text from LLM calls.
        """
        # --- Core Function Calls ---
        self.model_name = model_name
        self.base_config = base_config
        self._profiler = profiler_func
        self._builder = builder_func
        self._caller = caller_func
        self.dev_mode = dev_mode

        # --- Store Output Data ---
        # LLM reponse metadata & raw text (optional: dev_mode)
        self.response_data = {
            "suffixes": {"metadata": None, "response_text": None},
            "attribute_map": {"metadata": None, "response_text": None},
            "attribute_descriptions": {"metadata": None, "response_text": None},
            "normalized_schema": {"metadata": None, "response_text": None},
        }
        # Primary pipeline output
        self.dictionary = {
            "attribute_descriptions": None,
            "suffixes": None,
            "attribute_map": None,
            "normalized_schema": None,
            "final_attribute_profiles": None
        }

        # --- Prompt Arguments ---
        # Response times
        self.timings = {}
        
        # Standard prompt components
        self.allowed_characters = ['uppercase letters (A-Z)', 'lowercase_letters (a-z)', 'spaces']

    @time_step
    def generate_suffixes(self, dataframe: pd.DataFrame, dataset_context: str, prompt_format: str, generation_config: dict):
        """
        Step 1: Generates a standardized list of data suffixes from the dataset profile.
        """
        print("\nStep 1: Generating Suffixes...")
        
        # --- Define Prompt Components ---
        task = """
        Analyze the attribute profiles provided in 'attribute_profiles'. Each profile
        includes the original column name and technical data analysis.
        """
        # Call the profiler with only the technical data; no descriptions are available yet.
        attribute_profiles = self._profiler(dataframe)
        goal = """
        Based on your analysis of ALL the attribute profiles, identify common, reusable patterns
        in the names and data. Create a standardized list of suffixes that describe the type of data
        held by these attributes. For each suffix, determine the most appropriate logical data type
        from the list provided in 'data_types'.
        
        Return the output as a single JSON object with a key "suffixes" containing a list of objects.
        Each object in the list must contain three keys: "suffix", "logical_data_type", and "description".
        """
        example_structure = """
        {
          "suffixes": [
            {
              "suffix": "Identifier",
              "logical_data_type": "int",
              "description": "A unique numeric key for a record."
            },
            {
              "suffix": "Name",
              "logical_data_type": "str",
              "description": "The proper name or title of an entity."
            }
          ]
        }
        """
        data_types = ['str', 'int', 'float', 'bool', 'date', 'time', 'datetime', 'NoneType']

        suffix_formatting_rules = [
            "Suffixes should be formatted in Sentence Case.",
            "Each suffix should only be a single word constructed from the characters in allowed_characters'.",
            "All suffixes should be made up from full words. Acronyms, contractions and abbreviations" # cont.
            " should be expanded to full words (e.g. 'ID' will become 'Identifier')."
        ]

        # --- Build Prompt ---
        suffix_prompt = self._builder(
            output_format=prompt_format,
            task=task,
            attribute_profiles=attribute_profiles,
            goal=goal,
            example_structure=example_structure,
            data_types=data_types,
            suffix_formatting_rules=suffix_formatting_rules,
            allowed_characters=self.allowed_characters
        )
        
        # --- LLM Call Config ---
        suffix_config = generation_config.copy()
        suffix_config['response_schema'] = SuffixSchema
        
        # --- Send LLM Request ---
        response = self._caller(
            genai_model=self.model_name,
            prompt=suffix_prompt,
            config=suffix_config,
            return_df=True,
            return_metadata=self.dev_mode,
            return_response_text=self.dev_mode
        )

        if self.dev_mode:
            generated_suffixes, metadata, text = response
            self.response_data['suffixes']['metadata'] = metadata
            self.response_data['suffixes']['response_text'] = text
        else:
            generated_suffixes = response
        
        self.dictionary['suffixes'] = generated_suffixes
        print(f"{ind}> Complete – Suffixes stored.")
        return generated_suffixes

    @time_step
    def standardize_attribute_names(self, dataframe: pd.DataFrame, dataset_context: str, prompt_format: str, generation_config: dict):
        """
        Step 2: Generates standardized attribute names using the generated suffixes.
        """
        if self.dictionary['suffixes'] is None or self.dictionary['suffixes'].empty:
            print(f"{ind}> ERROR: Suffixes not found. Cannot proceed.")
            return None

        print("\nStep 2: Standardizing Attribute Names...")
        
        # --- Define Prompt Components ---
        task = """
        Analyze the attribute profiles provided in 'attribute_profiles'. Each profile
        includes the original column name and technical data analysis.
        """
        # Call the profiler with only the technical data.
        attribute_profiles = self._profiler(dataframe)
        
        goal = """
        Based on your analysis of ALL attribute profiles, generate new standardised attribute names
        that strictly adheres to the 'attribute_naming_rules' using the definitions outlined in
        'attribute_name_components'.
        """
        attribute_naming_rules = [
            "Attribute names are always formatted as: 'Subject Qualifier Suffix'.",
            "All attribute names should be made up from full words. Acronyms, contractions and abbreviations "
            "should be expanded to full words (e.g. 'ID' will become 'Identifier'.",
            "All names must be in Proper Case, with each word separated by a single space.",
            "All names must only contain characters found in 'allowed_characters'"
        ]
        attribute_name_components = """
        - **Subject**: This is the primary, descriptive characteristic of the attribute. It is **mandatory**.
        - **Qualifier**: A constraint or distinguishing characteristic. It is **only used** when two or more attributes would otherwise have the same Subject and Suffix.
        - **Suffix**: The data type of the attribute, chosen **strictly** from the list in 'approved_suffixes'. It should be omitted if the Subject or Qualifier already contains the exact suffix word.
        """

        approved_suffixes = self.dictionary['suffixes'].rename(columns={'suffix': 'column_name'}).to_dict(orient='records')

        # --- Build Prompt ---
        attribute_prompt = self._builder(
            output_format=prompt_format,
            task=task,
            attribute_profiles=attribute_profiles,
            goal=goal,
            attribute_naming_rules=attribute_naming_rules,
            attribute_name_components=attribute_name_components,
            approved_suffixes=approved_suffixes,
            allowed_characters=self.allowed_characters
        )
        
        # --- LLM Call Config ---
        attribute_config = generation_config.copy()
        attribute_config['response_schema'] = AttributeSchema

        # --- Send LLM Request ---
        response = self._caller(
            genai_model=self.model_name,
            prompt=attribute_prompt,
            config=attribute_config,
            return_df=True,
            return_metadata=self.dev_mode,
            return_response_text=self.dev_mode
        )

        if self.dev_mode:
            generated_attribute_names, metadata, text = response
            self.response_data['attribute_map']['metadata'] = metadata
            self.response_data['attribute_map']['response_text'] = text
        else:
            generated_attribute_names = response

        # --- Attribute Name Mapping Logic ---
        print(f"{ind}> INFO: Ensuring a complete 1-to-1 mapping for all original columns...")
        
        attr_names_map_dict = dict(zip(generated_attribute_names['old_column_name'], generated_attribute_names['new_attribute_name']))
        original_columns = dataframe.columns.tolist()
        
        complete_map_rows = []
        for col in original_columns:
            new_name = attr_names_map_dict.get(col, col.replace('_', ' ').title())
            complete_map_rows.append({'old_column_name': col, 'new_attribute_name': new_name})   
        attribute_map_df = pd.DataFrame(complete_map_rows)
        
        # --- Enforce uniqueness of new attribute names ---
        print(f"{ind}> INFO: Checking for and resolving duplicate standardized names...")
        
        # Identify which names are duplicates
        is_duplicate = attribute_map_df.duplicated(subset=['new_attribute_name'], keep=False)
        
        if is_duplicate.any():
            print(f"{ind}{ind}>> Duplicates found. Appending numeric suffixes...")
            
            # Create a cumulative count for each group of duplicated names
            counts = attribute_map_df.groupby('new_attribute_name').cumcount()
            
            # Create a suffix string (e.g., " 2", " 3"). The first item (count=0) gets no suffix.
            suffix = ' ' + (counts + 1).astype(str)
            suffix[counts == 0] = '' # Clear suffix for the first instance
            
            # Apply the suffix ONLY to the rows that were identified as duplicates
            attribute_map_df.loc[is_duplicate, 'new_attribute_name'] += suffix[is_duplicate]
        else:
            print(f"{ind}{ind}> No duplicates found.")

        self.dictionary['attribute_map'] = attribute_map_df
        print(f"{ind}> Complete – Foolproof attribute map stored.")
        return attribute_map_df

    @time_step
    def generate_descriptions(self, dataframe: pd.DataFrame, dataset_context: str, prompt_format: str, generation_config: dict):
        """
        Step 3: Generates a business description for each STANDARDIZED attribute.
        """
        if self.dictionary['attribute_map'] is None or self.dictionary['attribute_map'].empty:
            print(f"{ind}> ERROR: Attribute map not found. Cannot proceed.")
            return None
        
        print("\nStep 3: Generating Attribute Descriptions...")
        
        # --- Prepare Data with New Standardized Names ---
        attribute_map_df = self.dictionary['attribute_map']
        rename_dict = dict(zip(attribute_map_df.old_column_name, attribute_map_df.new_attribute_name))
        renamed_df = dataframe.rename(columns=rename_dict)

        technical_attribute_profiles = self._profiler(renamed_df)
        self.dictionary["final_attribute_profiles"] = technical_attribute_profiles
    
        # --- Define Prompt Components ---
        task = """
        Analyze the attribute profiles provided in the 'attribute_profiles' section. Each profile
        includes the attribute name and technical data analysis.
        """

        # 'technical_attribute_profiles' goes here in prompt order (moved above for clarity)
    
        goal = """
        Based on your analysis of ALL the profiles, write a clear, concise business description for
        each attribute.
        
        Return each attribute's name and description.
        """
        
        # --- Build Prompt ---
        description_prompt = self._builder(
            output_format=prompt_format,
            task=task,
            attribute_profiles=technical_attribute_profiles,
            goal=goal
        )
        
        # --- LLM Call Config ---
        description_config = generation_config.copy()
        description_config['response_schema'] = DescriptionSchema
        
        # --- Send LLM Request ---
        response = self._caller(
            genai_model=self.model_name,
            prompt=description_prompt,
            config=description_config,
            return_df=True,
            return_metadata=self.dev_mode,
            return_response_text=self.dev_mode
        )
        
        if self.dev_mode:
            generated_descriptions, metadata, text = response
            self.response_data['attribute_descriptions']['metadata'] = metadata
            self.response_data['attribute_descriptions']['response_text'] = text
        else:
            generated_descriptions = response
        
        # --- Finalize Descriptions ---
        print(f"{ind}> INFO: Enforcing standardized names in final descriptions output...")
        final_descriptions_df = pd.merge(
            attribute_map_df[['new_attribute_name']].rename(columns={'new_attribute_name': 'attribute_name'}),
            generated_descriptions,
            on='attribute_name',
            how='left'
        )
        # Fill any descriptions the LLM may have missed.
        final_descriptions_df['business_description'] = final_descriptions_df['business_description'].fillna("[MISSING DESCRIPTION]")
        
        self.dictionary['attribute_descriptions'] = final_descriptions_df
        print(f"{ind}> Complete – attribute descriptions stored.")
        return final_descriptions_df

    @time_step
    def normalize_schema(self, dataframe: pd.DataFrame, dataset_context: str, prompt_format: str, generation_config: dict):
        """
        Step 4: Produces a 3NF relational schema using the standardized attributes and descriptions.
        """
        if self.dictionary['attribute_map'] is None or self.dictionary['attribute_map'].empty:
            print(f"{ind}> ERROR: Attribute map not found. Cannot proceed.")
            return None
        
        print("\nStep 4: Normalizing Schema to 3NF...")
        
        # --- Prepare Data with New Names ---
        attribute_map_df = self.dictionary['attribute_map']
        rename_dict = dict(zip(attribute_map_df.old_column_name, attribute_map_df.new_attribute_name))
        renamed_df = dataframe.rename(columns=rename_dict)
        
        # --- Prepare Descriptions for Profiler ---
        descriptions_df = self.dictionary.get('attribute_descriptions')
        profiler_desc_df = None
        if descriptions_df is not None:
            # Ignore descriptions that are only a hyphen before passing to profiler
            profiler_desc_df = descriptions_df[descriptions_df['business_description'].str.strip() != '-'].copy()

        attribute_profiles_ext = self._profiler(renamed_df, profiler_desc_df)
        self.dictionary["final_attribute_profiles"] = attribute_profiles_ext
        
        # --- Define Prompt Components ---
        task = """
        Analyze the attribute profiles provided in 'attribute_profiles'. Each profile
        includes the column name, a business description, and technical data analysis.
        """
        
        # 'attribute_profiles_ext' goes here in prompt order
        
        goal = """
        Based on your analysis of ALL attribute profiles, produce a third normal form (3NF) relational
        data schema for this dataset. The entities within the schema should have attributes assigned as primary
        and foreign keys where appropriate.
        
        When naming entities, follow the rules in 'entity_naming_rules'.
        """
        entity_naming_rules = [
            "All attribute names should be made up from full words. Acronyms, contractions and abbreviations "
            "should be expanded to full words (e.g. 'ID' will become 'Identifier'.",
            "All entity names must be in Proper Case, with each word separated by a single space.",
            "All entity names must only contain characters found in 'allowed_characters'"
        ]
        
        normalization_prompt = self._builder(
            output_format=prompt_format,
            task=task,
            attribute_profiles=attribute_profiles_ext,
            goal=goal,
            entity_naming_rules=entity_naming_rules,
            allowed_characters=self.allowed_characters
        )

        # --- LLM Call Config ---
        draft_normalization_config = generation_config.copy()
        draft_normalization_config['response_schema'] = DraftNormalizedSchema

        # --- Send LLM Request (Unique) ---
        # This step always needs the raw text for two-pass validation
        response = self._caller(
            genai_model=self.model_name,
            prompt=normalization_prompt,
            config=draft_normalization_config,
            return_df=False,
            return_metadata=self.dev_mode,
            return_response_text=True # raw text
        )

        if self.dev_mode:
            # Returns (parsed_obj, metadata, text)
            draft_schema, metadata, draft_schema_text = response
            self.response_data['normalized_schema']['metadata'] = metadata
            self.response_data['normalized_schema']['response_text'] = draft_schema_text
        else:
            # Returns (parsed_obj, text)
            draft_schema, draft_schema_text = response
            self.response_data['normalized_schema']['response_text'] = draft_schema_text

        if not draft_schema or not draft_schema_text:
            print(f"{ind}> ERROR: Failed to generate a draft schema from the LLM.")
            return None

        # --- Multi-pass validation logic ---
        suffixes_set = {s.lower() for s in self.dictionary["suffixes"]['suffix'].tolist()}
        newly_found_suffixes = {
            attr.attribute_name.suffix.lower() 
            for entity in draft_schema.entities 
            for attr in entity.attributes 
            if attr.attribute_name.suffix
        }        
        if newly_found_suffixes - suffixes_set:
            print(f"{ind}> INFO: Found and added new suffixes: {newly_found_suffixes - suffixes_set}")
            suffixes_set.update(newly_found_suffixes)
            self.dictionary["suffixes"] = pd.DataFrame(list(suffixes_set), columns=['suffix'])
        try:
            print(f"{ind}> Performing final validation with refined suffix list...")
            final_validated_schema = NormalizedSchema.model_validate_json(
                draft_schema_text, context={"allowed_suffixes": suffixes_set}
            )
            final_schema_df = self._flatten_schema_to_df(final_validated_schema)
            self.dictionary['normalized_schema'] = final_schema_df
            return self.dictionary['normalized_schema']
        except ValidationError as e:
            print(f"{ind}> ERROR: Final validation failed (after refining suffixes).\n{e}")
            return None

    def run_full_pipeline(
        self,
        dataframe: pd.DataFrame, dataset_summary: str,
        config_overrides: dict, prompt_format: str = 'xml',
        normalize: bool = False
    ):
        """
        Executes the entire data modelling pipeline sequentially.
        """
        # Set LLM generation configs
        final_config = self.base_config.copy()
        final_config.update(config_overrides)
        
        # Record start time and clear previous timings
        pipeline_start_time = time.time()
        self.timings.clear()

        # Step 1: Generate Suffixes
        if self.generate_suffixes(dataframe, dataset_summary, prompt_format, final_config) is None:
            print("\nERROR: Pipeline halted at Step 1.")
            return None, self.timings # return timings even on failure

        # Step 2: Standardize Attribute Names
        if self.standardize_attribute_names(dataframe, dataset_summary, prompt_format, final_config) is None:
            print("\nERROR: Pipeline halted at Step 2.")
            return None, self.timings
        
        # Step 3: Generate Descriptions for Standardized Names
        if self.generate_descriptions(dataframe, dataset_summary, prompt_format, final_config) is None:
            print("\nERROR: Pipeline halted at Step 3.")
            return None, self.timings

        # Step 4 (Optional): Normalize Schema
        if normalize:
            if self.normalize_schema(dataframe, dataset_summary, prompt_format, final_config) is None:
                print("\nERROR: Pipeline halted at Step 4.")
                return None, self.timings
        else:
            print("\n--- Pipeline Complete – Normalization skipped ---\n")

        # --- Calculate and display total time ---
        total_duration = round(time.time() - pipeline_start_time, 2)
        self.timings['total_run_time'] = total_duration
        print(f"\nTotal Pipeline Time: {total_duration}s")

        return self.dictionary, self.timings
    
    def _flatten_schema_to_df(self, schema_object: "NormalizedSchema") -> pd.DataFrame:
        """
        Converts the final NormalizedSchema object into a flattened DataFrame.
        """
        flattened_data = []
        schema_dict = schema_object.model_dump()

        for entity in schema_dict['entities']:
            entity_name = entity['entity_name']
            for attribute in entity['attributes']:
                row = {
                    'Entity Name': entity_name,
                    'Attribute Name': attribute['attribute_name']['build_full_name'],
                    'Attribute Key': attribute['attribute_key'],
                    'Subject': attribute['attribute_name']['subject'],
                    'Qualifier': attribute['attribute_name']['qualifier'],
                    'Suffix': attribute['attribute_name']['suffix']
                }
                flattened_data.append(row)
        
        return pd.DataFrame(flattened_data)

### 5. Toolkit Runner Functions

In [82]:
# --- 5.1 Runner Utility Functions ---

def create_compound_view(results: dict, normalize: bool):
    """
    Creates a flattened dataframe that combines profiles and descriptions.
    """
    if not results.get("final_attribute_profiles"):
        print(f"{ind}> INFO: No profiling data found to create a compound view.")
        return None

    # Flatten the list of profile dictionaries into a dataframe
    profiles_df = pd.json_normalize(results["final_attribute_profiles"])
    
    if 'column_name' in profiles_df.columns:
        profiles_df = profiles_df.rename(columns={'column_name': 'attribute_name'})
        
    descriptions_df = results.get('attribute_descriptions')
    if descriptions_df is not None:
        # Create a simple Series to map attribute names to their descriptions
        desc_map = pd.Series(
            descriptions_df.business_description.values, 
            index=descriptions_df.attribute_name
        )
        # Use the map to create/update business_description column
        profiles_df['business_description'] = profiles_df['attribute_name'].map(desc_map)

    # Drop redundant description column
    if 'description' in profiles_df.columns:
        profiles_df = profiles_df.drop(columns=['description'])

    # Normalize & return the compound view
    if normalize and results.get("normalized_schema") is not None:
        schema_df = results["normalized_schema"]
        merged_df = pd.merge(
            schema_df[['Entity Name', 'Attribute Name']],
            profiles_df,
            left_on='Attribute Name',
            right_on='attribute_name',
            how='left'
        )
        entity_dfs = {
            entity_name: entity_group.drop(columns=['Entity Name', 'attribute_name'])
                                     .rename(columns={'Attribute Name': 'attribute_name'})
                                     .set_index('attribute_name')
            for entity_name, entity_group in merged_df.groupby('Entity Name')
        }
        return entity_dfs
    else:
        return profiles_df.set_index('attribute_name')

# --- Save Metadata ---
def _save_run_metadata(all_results: dict, batch_output_dir: Path, context_name: str):
    """
    Processes the final results dictionary to extract, flatten, and save
    performance and cost metadata to a single CSV file for the batch run.
    """
    print("\nProcessing and saving performance metadata...")
    metadata_records = []

    # The `all_results` dictionary key is a tuple:
    # (dataset_name, model_name, prompt_format, config_name)
    for params, result_data in all_results.items():
        dataset, model, prompt_format, config = params

        # Base record with run parameters
        record = {
            'run_context': context_name,
            'dataset': dataset,
            'model': model,
            'prompt_format': prompt_format,
            'config': config,
            'status': result_data.get('status', 'SUCCESS')
        }

        if record['status'] == 'FAILED':
            record['error'] = result_data.get('error', 'Unknown error')
            metadata_records.append(record)
            continue  # Move to the next result

        # --- Process Timings ---
        timings = result_data.get('timings', {})
        for step, duration in timings.items():
            # Add a 'time_' prefix for clarity
            record[f'time_{step}'] = duration

        # --- Process Token Usage ---
        grand_total_prompt_tokens = 0
        grand_total_completion_tokens = 0
        grand_total_tokens = 0

        metadata = result_data.get('results_metadata', {})
        for step, step_meta in metadata.items():
            if step_meta and 'metadata' in step_meta and step_meta['metadata']:
                # Prompt tokens from the pre-call count
                prompt_tokens = step_meta['metadata'].get('prompt_tokens', 0)
                record[f'tokens_prompt_{step}'] = prompt_tokens
                grand_total_prompt_tokens += prompt_tokens

                # Detailed tokens from the response metadata
                usage = step_meta['metadata'].get('response_usage_metadata')
                if usage:
                    # --- FIXED CODE BLOCK ---
                    # Use getattr() for safe attribute access on the object
                    completion_tokens = getattr(usage, 'candidates_token_count', 0)
                    step_total_tokens = getattr(usage, 'total_token_count', 0)

                    record[f'tokens_completion_{step}'] = completion_tokens
                    record[f'tokens_total_{step}'] = step_total_tokens

                    grand_total_completion_tokens += completion_tokens
                    grand_total_tokens += step_total_tokens
                    # --- END OF FIX ---

        record['grand_total_prompt_tokens'] = grand_total_prompt_tokens
        record['grand_total_completion_tokens'] = grand_total_completion_tokens
        record['grand_total_tokens'] = grand_total_tokens

        metadata_records.append(record)

    if not metadata_records:
        print("No metadata records found to save.")
        return

    # Create DataFrame and save to CSV
    metadata_df = pd.DataFrame(metadata_records)

    # Define a logical column order for the output CSV
    param_cols = ['run_context', 'dataset', 'model', 'prompt_format', 'config', 'status', 'error']
    time_cols = sorted([col for col in metadata_df.columns if col.startswith('time_')])
    total_token_cols = sorted([col for col in metadata_df.columns if col.startswith('grand_total_')])
    step_token_cols = sorted([col for col in metadata_df.columns if col.startswith('tokens_')])
    
    # Ensure 'error' column exists if there were failures, otherwise it's omitted
    final_cols = [col for col in param_cols if col in metadata_df.columns]
    final_cols += time_cols + total_token_cols + step_token_cols
    
    metadata_df = metadata_df.reindex(columns=final_cols)

    output_path = batch_output_dir / f"{context_name}_performance_metadata.csv"
    try:
        metadata_df.to_csv(output_path, index=False)
        print(f"Performance and cost metadata saved to: '{output_path}'")
    except Exception as e:
        print(f"FAILED to save metadata. Error: {e}")

In [83]:
# --- 5.2 Dataset Input Validation ---

class DatasetConfig(BaseModel):
    """
    Validate and manage dataset configurations.
    """
    file_name: str
    summary: str
    normalize: bool = False
    prompt_format: Literal['xml', 'json', 'markdown', 'plaintext'] = 'xml'
    generation_config: Optional[dict] = {}
    
    @field_validator('prompt_format', mode='before')
    @classmethod
    def validate_prompt_format(cls, v, info):
        """
        Validate the prompt_format, defaulting to 'xml' if invalid.
        """
        allowed_formats = {'xml', 'json', 'markdown', 'plaintext'}
        if v is not None and v not in allowed_formats:
            file_name = info.data.get('file_name', 'Unknown File')
            print(f"{ind}> INFO: Invalid 'prompt_format' ('{v}') for '{file_name}'"
                  "      >> Defaulting to XML"
                 )
            return 'xml' # return default value
        return v

In [84]:
# --- 5.3 Dataset Processing ---

def process_dataset(
    schema_manager: "SchemaManager",
    file_path: Path,
    dataset_summary: str,
    output_dir: Path,
    generation_config: dict,
    prompt_format: str,
    normalize: bool
) -> dict:
    """
    Runs a single pipeline instance for a given dataset and configuration.
    """
    # The main loop now handles the progress counter print statement
    
    # --- Load Data ---
    try:
        data = dataset_reader(file_path, show_progress=False)
    except FileNotFoundError:
        print(f"   > ERROR: File not found at {file_path}. Skipping.")
        return {}

    # --- Run the pipeline ---
    pipeline_results, timings = schema_manager.run_full_pipeline(
        dataframe=data,
        dataset_summary=dataset_summary,
        prompt_format=prompt_format,
        normalize=normalize,
        config_overrides=generation_config
    )
    
    if not pipeline_results:
        return {}

    # --- Structure and Save Results ---
    results = {
        'processed_data': pipeline_results,
        'timings': timings,
        'results_metadata': schema_manager.response_data if schema_manager.dev_mode else {}
    }

    # Create compound view forimmediate display/checking
    compound_views = create_compound_view(results['processed_data'], normalize)
    if compound_views is not None:
         results['processed_data']['compound_views'] = compound_views

    save_output_csv(
        output_dir=output_dir,
        model_name=schema_manager.model_name,
        prompt_format=prompt_format,
        config_name=generation_config.get('name', 'custom_config'),
        results_dict=results # Pass the entire results dictionary
    )
    
    print(f"{ind}> Save complete.")
    
    return results

    # --- Create and Display Compound View ---
    print("Generating Compound Attribute View(s)...")
    
    # Pass the nested dictionary to the view creator
    compound_views = create_compound_view(results['processed_data'], normalize)
    
    if isinstance(compound_views, dict): # Case for normalize=True
        for entity_name, entity_df in compound_views.items():
            print(f"\nEntity: {entity_name}")
            display(entity_df)
    elif compound_views is not None: # Case for normalize=False
        display(compound_views)
            
    # Add the compound views to the nested processed_data dictionary
    results['processed_data']['compound_views'] = compound_views

    # --- Save Results ---
    dataset_output_dir = output_dir / f"{file_path.stem}_results"
    dataset_output_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n{ind}> Saving results to: {dataset_output_dir}")
    # The save function is robust enough to handle the new nested structure
    save_output_csv(
        output_dir=dataset_output_dir,
        # 'results' contains 'processed_data' and 'results_metadata' as top-level keys
        # The function will iterate and save DataFrames from 'processed_data'
        final_outputs=results 
    )
    print(f"{ind}> Save complete.")
    
    return results

In [85]:
# --- 5.4 Core Runner Function ---

def main(
    data_dir: str,
    output_dir: str,
    datasets: list,
    llm_models: list,
    prompt_formats: list,
    generation_configs: list,
    context_name: str = 'toolkit_run',
    dev_mode: bool = False,
    override_prompt_format: Optional[str] = None,
    override_generation_config: Optional[dict] = None
) -> dict:
    """
    Main function to orchestrate the processing of multiple datasets across
    various models, prompt formats, and generation configurations.
    """
    # Determine which configurations to use based on overrides
    p_formats = [override_prompt_format] if override_prompt_format else prompt_formats
    g_configs = [override_generation_config] if override_generation_config else generation_configs
    
    # Create a unique, timestamped folder for the entire batch run
    timestamp_str = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    batch_folder_name = f"{context_name}_{timestamp_str}"
    batch_output_dir = output_dir / batch_folder_name
    batch_output_dir.mkdir(parents=True, exist_ok=True)
    print(f"All outputs for this batch run will be saved in: '{batch_output_dir}'")

    # --- Define Base Model Config ---
    base_model_config = {
        'system_instruction': "You are a data modelling professional.",
        'temperature': 0.2,
        'top_p': 0.95,
        'top_k': 20,
        'response_mime_type': "application/json"
    }
    
    all_results = {}
    total_runs = len(datasets) * len(llm_models) * len(p_formats) * len(g_configs)
    current_run = 0
    
    # --- Main Experiment Loop ---
    for dataset_info in datasets:
        try:
            dataset_config = DatasetConfig(**dataset_info)
            file_path = data_dir / dataset_config.file_name
            
            # Create a dedicated folder for the current dataset
            dataset_output_dir = batch_output_dir / file_path.stem
            dataset_output_dir.mkdir(exist_ok=True)

        except (ValidationError, FileNotFoundError) as e:
            print(f"> ERROR: Skipping invalid dataset '{dataset_info.get('file_name')}'.\n  Reason: {e}")
            continue

        for model_name in llm_models:
            # Instantiate SchemaManager (once per llm model)
            schema_manager = SchemaManager(
                model_name=model_name,
                base_config=base_model_config,
                profiler_func=dataset_profiler,
                builder_func=prompt_builder,
                caller_func=generate_output,
                dev_mode=dev_mode
            )
            
            for prompt_format in p_formats:
                for gen_config in g_configs:
                    current_run += 1
                    try:
                        print(f"\n{'='*25} RUN {current_run}/{total_runs} {'='*25}")
                        print(f"Dataset: {file_path.name}\nModel: {model_name}\nPrompt Format: {prompt_format}\nConfig: {gen_config.get('name', 'custom_config')}")
                        
                        # Use the config from the dataset dict as a base, then apply the loop's config
                        final_gen_config = dataset_config.generation_config.copy()
                        final_gen_config.update(gen_config)
                        
                        results = process_dataset(
                            schema_manager=schema_manager,
                            file_path=file_path,
                            dataset_summary=dataset_config.summary,
                            output_dir=dataset_output_dir,
                            generation_config=final_gen_config,
                            prompt_format=prompt_format,
                            normalize=dataset_config.normalize
                        )
                        
                        # Store results with a unique key
                        result_key = (file_path.stem, model_name, prompt_format, gen_config.get('name', 'custom'))
                        all_results[result_key] = results
                     
                    except Exception as e:
                        # If any error occurs during the run, log it and continue
                        print(f"\n--- RUN {current_run}/{total_runs} FAILED ---")
                        print(f"Dataset: {file_path.name}")
                        print(f"Model: {model_name}")
                        print(f"Config: {gen_config.get('name', 'custom_config')}")
                        print(f"Error: {e}\n")
                        
                        # Store the failure in the results dictionary
                        result_key = (file_path.stem, model_name, prompt_format, gen_config.get('name', 'custom'))
                        all_results[result_key] = {'status': 'FAILED', 'error': str(e)}
                        continue

    print(f"\n{'='*25} BATCH RUN FINISHED {'='*25}")
    print(f"Successfully completed {len(all_results)} of {total_runs} planned runs")

    # Save metadata
    _save_run_metadata(all_results, batch_output_dir, context_name)
    
    return all_results

## Run the Toolkit

### 6. Experiment Setup

In [88]:
# --- 6.1 Set Up Run Parameters ---

# --- Directory Configuration ---
proj_dir = Path('/Users/nickballingall/Library/CloudStorage/OneDrive-UniversityofStrathclyde/MSc FInTech/Uni Projects/Finance – Lloyds/Automation Toolkit Project')
data_dir = proj_dir / 'Input Data'
output_dir = proj_dir / 'Output Data'

# --- LLM Call Option Sets ---
llm_models = ['gemini-2.5-flash-lite', 'gemini-2.5-flash', 'gemini-2.5-pro' ]
prompt_formats = ['xml', 'json', 'markdown']
generation_configs = [
    {
        'name': 'Factual',
        'system_instruction': "You are a data modelling professional.",
        'temperature': 0.3, 'top_p': 0.95, 'top_k': 20,
        'response_mime_type': "application/json"
    },
    {
        'name': 'Balanced',
        'system_instruction': "You are a data modelling professional.",
        'temperature': 1.0, 'top_p': 1.0,
        'response_mime_type': "application/json"
    }
]

# --- Dataset Parameters ---
"""
NOTE: The 'generation_config' and 'prompt_format' here now act as a base,
which can be overridden by the loop or main() function arguments
"""
test_datasets = [
    {
        'file_name': 'uber_rides_data_exp.csv',
        'summary': "A comprehensive dataset of Uber ride-sharing data.",
        'normalize': True,
        'prompt_format': 'xml', # base prompt format
        'generation_config': {} # base config is empty, will use loop's config
    },
    {
        'file_name': 'hearing_wellbeing_survey_report.csv',
        'summary': "Captures public perceptions and awareness around hearing health.",
        'normalize': True,
        'prompt_format': 'xml',
        'generation_config': {}
    },
    {
        'file_name': 'nasa_panetary_systems_composite_data.csv',
        'summary': "Planetary Systems Composite Parameters (PSCP)",
        'normalize': True,
        'prompt_format': 'xml',
        'generation_config': {}
    }
]

In [89]:
# --- 6.2 Run the Main Loop ---

# --- Experiment A ---
all_experiment_results = {}

for i in range(1, 3): # 3 experiments
    current_context_name = f'EXP_A_{i}' # set experiment name
    
    print('='*64)
    print(f"{'='*25} {current_context_name.upper()} {'='*25}")
    print('='*64)
    
    experiment_results = main(
        data_dir=data_dir,
        output_dir=output_dir,
        datasets=test_datasets,
        llm_models=llm_models,
        prompt_formats=prompt_formats,
        generation_configs=generation_configs,
        context_name=current_context_name, # update experiment name sequentially
        dev_mode=True,
    )
    
    # Store results
    all_experiment_results[current_context_name] = experiment_results

========================= EXP_A_1 =========================
All outputs for this batch run will be saved in: '/Users/nickballingall/Library/CloudStorage/OneDrive-UniversityofStrathclyde/MSc FInTech/Uni Projects/Finance – Lloyds/Automation Toolkit Project/Output Data/EXP_A_1_2025-10-12_20-14-26'

========================= RUN 1/54 =========================
Dataset: uber_rides_data_exp.csv
Model: gemini-2.5-flash-lite
Prompt Format: xml
Config: Factual

Step 1: Generating Suffixes...
   > Profiling dataset...
   > Complete – Suffixes stored.
   > 'generate_suffixes' completed in 3.37s

Step 2: Standardizing Attribute Names...
   > Profiling dataset...
   > INFO: Ensuring a complete 1-to-1 mapping for all original columns...
   > INFO: Checking for and resolving duplicate standardized names...
      > No duplicates found.
   > Complete – Foolproof attribute map stored.
   > 'standardize_attribute_names' completed in 2.36s

Step 3: Generating Attribute Descriptions...
   > Profiling datase

In [90]:
# --- Experiment B ---
all_experiment_results = {}

for i in range(1, 4):
    current_context_name = f'EXP_B_{i}'
    
    print('='*64)
    print(f"{'='*25} {current_context_name.upper()} {'='*25}")
    print('='*64)
    
    experiment_results = main(
        data_dir=data_dir,
        output_dir=output_dir,
        datasets=test_datasets,
        llm_models=llm_models,
        prompt_formats=prompt_formats,
        generation_configs=generation_configs,
        context_name=current_context_name,
        dev_mode=True,
    )
    
    # Store results
    all_experiment_results[current_context_name] = experiment_results

========================= EXP_B_1 =========================
All outputs for this batch run will be saved in: '/Users/nickballingall/Library/CloudStorage/OneDrive-UniversityofStrathclyde/MSc FInTech/Uni Projects/Finance – Lloyds/Automation Toolkit Project/Output Data/EXP_B_1_2025-10-13_01-17-12'

========================= RUN 1/54 =========================
Dataset: uber_rides_data_exp.csv
Model: gemini-2.5-flash-lite
Prompt Format: xml
Config: Factual

Step 1: Generating Suffixes...
   > Profiling dataset...
   > Complete – Suffixes stored.
   > 'generate_suffixes' completed in 3.22s

Step 2: Standardizing Attribute Names...
   > Profiling dataset...
   > INFO: Ensuring a complete 1-to-1 mapping for all original columns...
   > INFO: Checking for and resolving duplicate standardized names...
      > No duplicates found.
   > Complete – Foolproof attribute map stored.
   > 'standardize_attribute_names' completed in 1.98s

Step 3: Generating Attribute Descriptions...
   > Profiling datase

In [91]:
# --- Experiment C ---
all_experiment_results = {}

for i in range(1, 4):
    current_context_name = f'EXP_C_{i}'
    
    print('='*64)
    print(f"{'='*25} {current_context_name.upper()} {'='*25}")
    print('='*64)
    
    experiment_results = main(
        data_dir=data_dir,
        output_dir=output_dir,
        datasets=test_datasets,
        llm_models=llm_models,
        prompt_formats=prompt_formats,
        generation_configs=generation_configs,
        context_name=current_context_name,
        dev_mode=True,
    )
    
    # Store results
    all_experiment_results[current_context_name] = experiment_results

========================= EXP_C_1 =========================
All outputs for this batch run will be saved in: '/Users/nickballingall/Library/CloudStorage/OneDrive-UniversityofStrathclyde/MSc FInTech/Uni Projects/Finance – Lloyds/Automation Toolkit Project/Output Data/EXP_C_1_2025-10-13_09-08-02'

========================= RUN 1/54 =========================
Dataset: uber_rides_data_exp.csv
Model: gemini-2.5-flash-lite
Prompt Format: xml
Config: Factual

Step 1: Generating Suffixes...
   > Profiling dataset...
   > Complete – Suffixes stored.
   > 'generate_suffixes' completed in 2.86s

Step 2: Standardizing Attribute Names...
   > Profiling dataset...
   > INFO: Ensuring a complete 1-to-1 mapping for all original columns...
   > INFO: Checking for and resolving duplicate standardized names...
      > No duplicates found.
   > Complete – Foolproof attribute map stored.
   > 'standardize_attribute_names' completed in 2.35s

Step 3: Generating Attribute Descriptions...
   > Profiling datase